In [1]:
# =========================================================
# user settings
# =========================================================
BATCH_SIZE = 64

rcm_var = "tas"
gcm_name = "CanESM2"
rcm_name = "RCA4"
grid = "NAM-44i"
rcm_product = "raw"
factor = 4

DATA_ROOT = "/projects/sds-lab/Shuochen/downscaling/preprocessed"


# =========================================================
# imports
# =========================================================
import math
import os

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


# =========================================================
# paths
# =========================================================
exp_folder_name = os.path.join(
    DATA_ROOT,
    f"{rcm_var}.{gcm_name}.{rcm_name}.day.{grid}.{rcm_product}.GCM_RCM",
)

input_file = "low_res.pth"
hr_mask_file = "high_res_mask.pth"
target_file = "high_res.pth"


# =========================================================
# helpers
# =========================================================
def assert_finite(tensor, name):
    if not torch.isfinite(tensor).all():
        raise ValueError(f"{name} contains NaN or inf values.")


def match_sample_dim(tensor, n_samples, name):
    if tensor.shape[0] == n_samples:
        return tensor

    if tensor.shape[0] == 1:
        print(f"{name} has one sample; expanding to {n_samples} samples.")
        return tensor.expand(n_samples, -1, -1, -1)

    raise ValueError(
        f"{name} has incompatible sample dimension: "
        f"{tensor.shape[0]} vs expected {n_samples}"
    )


START_YEAR = 1951
TRAIN_END_YEAR = 2005
VAL_RANGES = [
    (2006, 2099),
    (2006, 2040),
    (2041, 2070),
    (2071, 2099),
]


def is_leap(year):
    return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)


def days_between_years(start_year, end_year_inclusive):
    if end_year_inclusive < start_year:
        return 0
    total = 0
    for year in range(start_year, end_year_inclusive + 1):
        total += 366 if is_leap(year) else 365
    return total


def year_range_indices(start_year, end_year):
    if end_year < start_year:
        raise ValueError("end_year must be >= start_year")

    if gcm_name == "CanESM2":
        start_idx = (start_year - START_YEAR) * 365
        end_idx = (end_year - START_YEAR + 1) * 365
    elif gcm_name == "EC-EARTH":
        start_idx = days_between_years(START_YEAR, start_year - 1)
        end_idx = days_between_years(START_YEAR, end_year)
    else:
        raise ValueError(f"Year split is not defined for gcm_name={gcm_name}")

    return start_idx, end_idx


def mse_to_rmse(mse):
    return mse ** 0.5


def ssim_batch(pred, target, data_range, window_size=11):
    pred = pred.float()
    target = target.float()
    data_range = max(float(data_range), 1e-12)
    c1 = (0.01 * data_range) ** 2
    c2 = (0.03 * data_range) ** 2
    pad = window_size // 2

    pred_pad = F.pad(pred, (pad, pad, pad, pad), mode="reflect")
    target_pad = F.pad(target, (pad, pad, pad, pad), mode="reflect")
    mu_pred = F.avg_pool2d(pred_pad, window_size, stride=1)
    mu_target = F.avg_pool2d(target_pad, window_size, stride=1)

    mu_pred_sq = mu_pred ** 2
    mu_target_sq = mu_target ** 2
    mu_pred_target = mu_pred * mu_target

    sigma_pred_sq = F.avg_pool2d(pred_pad * pred_pad, window_size, stride=1) - mu_pred_sq
    sigma_target_sq = F.avg_pool2d(target_pad * target_pad, window_size, stride=1) - mu_target_sq
    sigma_pred_target = F.avg_pool2d(pred_pad * target_pad, window_size, stride=1) - mu_pred_target

    ssim_map = ((2 * mu_pred_target + c1) * (2 * sigma_pred_target + c2)) / (
        (mu_pred_sq + mu_target_sq + c1) * (sigma_pred_sq + sigma_target_sq + c2)
    )
    return torch.clamp(ssim_map, -1.0, 1.0).mean(dim=(1, 2, 3))


# =========================================================
# load data
# =========================================================
X = torch.load(os.path.join(exp_folder_name, input_file)).float()
hr_mask = torch.load(os.path.join(exp_folder_name, hr_mask_file)).float()
y = torch.load(os.path.join(exp_folder_name, target_file)).float()

hr_mask = match_sample_dim(hr_mask, y.shape[0], "hr_mask")
hr_mask = (hr_mask > 0.5).float()

if X.shape[0] != y.shape[0]:
    raise ValueError(f"Sample mismatch: X has {X.shape[0]}, y has {y.shape[0]}")

if hr_mask.shape != y.shape:
    raise ValueError(f"HR mask shape {hr_mask.shape} does not match y shape {y.shape}")

if y.shape[-2] != factor * X.shape[-2]:
    raise ValueError(
        f"Height mismatch: HR height {y.shape[-2]} != "
        f"{factor} * LR height {X.shape[-2]}"
    )

if y.shape[-1] != factor * X.shape[-1]:
    raise ValueError(
        f"Width mismatch: HR width {y.shape[-1]} != "
        f"{factor} * LR width {X.shape[-1]}"
    )

assert_finite(X, "X")
assert_finite(hr_mask, "hr_mask")
assert_finite(y, "y")

print("Loaded shapes:")
print("X      :", X.shape)
print("hr_mask:", hr_mask.shape)
print("y      :", y.shape)


# =========================================================
# train split for normalization
# =========================================================
train_start_idx, train_end_idx = year_range_indices(START_YEAR, TRAIN_END_YEAR)
X_train = X[train_start_idx:train_end_idx]
y_train = y[train_start_idx:train_end_idx]

if X_train.shape[0] == 0:
    raise ValueError("Training split is empty.")

y_mean = y_train.mean()
y_std = y_train.std()
if y_std.item() == 0:
    raise ValueError("Cannot normalize with zero y standard deviation.")

print("Training normalization split:")
print(f"years      : {START_YEAR}-{TRAIN_END_YEAR}")
print("X_train   :", X_train.shape)
print("y_train   :", y_train.shape)
print("y_mean    :", y_mean.item())
print("y_std     :", y_std.item())


# =========================================================
# bilinear baseline over validation periods
# =========================================================
def period_tensors(start_year, end_year):
    start_idx, end_idx = year_range_indices(start_year, end_year)
    if end_idx > X.shape[0]:
        raise ValueError(
            f"Range {start_year}-{end_year} ends at index {end_idx}, "
            f"but data has only {X.shape[0]} samples."
        )
    if end_idx <= start_idx:
        raise ValueError(f"Validation range is empty: {start_year}-{end_year}")

    return (
        X[start_idx:end_idx].contiguous(),
        y[start_idx:end_idx].contiguous(),
        hr_mask[start_idx:end_idx].contiguous(),
    )


def evaluate_bilinear_period(start_year, end_year):
    X_period, y_period, mask_period = period_tensors(start_year, end_year)
    loader = DataLoader(
        TensorDataset(X_period, y_period, mask_period),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    target_range = float((y_period.max() - y_period.min()).item())
    if target_range <= 0:
        target_range = 1.0

    normalized_sse = 0.0
    normalized_count = 0
    full_sse = 0.0
    full_count = 0
    land_sse = 0.0
    land_count = 0.0
    ssim_sum = 0.0
    ssim_count = 0

    with torch.no_grad():
        for X_batch, y_batch, mask_batch in loader:
            pred_batch = F.interpolate(
                X_batch,
                size=y_batch.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

            se = (pred_batch - y_batch) ** 2
            se_n = ((pred_batch - y_mean) / y_std - (y_batch - y_mean) / y_std) ** 2

            normalized_sse += se_n.sum().item()
            normalized_count += se_n.numel()
            full_sse += se.sum().item()
            full_count += se.numel()
            land_sse += (se * mask_batch).sum().item()
            land_count += mask_batch.sum().item()

            batch_ssim = ssim_batch(pred_batch, y_batch, target_range)
            ssim_sum += batch_ssim.sum().item()
            ssim_count += batch_ssim.numel()

    if full_count <= 0:
        raise ValueError(f"Range {start_year}-{end_year} has no pixels.")
    if land_count <= 0:
        raise ValueError(f"Range {start_year}-{end_year} has no land pixels.")

    full_mse = full_sse / full_count
    land_mse = land_sse / land_count
    psnr = float("inf") if full_mse == 0 else 10.0 * math.log10((target_range ** 2) / full_mse)

    return {
        "years": f"{start_year}-{end_year}",
        "samples": X_period.shape[0],
        "normalized_mse": normalized_sse / normalized_count,
        "physical_mse_full_image": full_mse,
        "physical_mse_land_only": land_mse,
        "full_rmse": mse_to_rmse(full_mse),
        "land_rmse": mse_to_rmse(land_mse),
        "psnr": psnr,
        "ssim": ssim_sum / ssim_count,
    }


bilinear_results = [
    evaluate_bilinear_period(start_year, end_year)
    for start_year, end_year in VAL_RANGES
]

print("")
print("Bilinear low_res -> high_res validation baseline")
print(
    f"{'years':<11} {'samples':>8} {'normalized_mse':>15} "
    f"{'full_mse':>10} {'land_mse':>10} {'full_rmse':>10} "
    f"{'land_rmse':>10} {'PSNR':>10} {'SSIM':>10}"
)
for row in bilinear_results:
    print(
        f"{row['years']:<11} "
        f"{row['samples']:>8d} "
        f"{row['normalized_mse']:>15.7f} "
        f"{row['physical_mse_full_image']:>10.3f} "
        f"{row['physical_mse_land_only']:>10.3f} "
        f"{row['full_rmse']:>10.3f} "
        f"{row['land_rmse']:>10.3f} "
        f"{row['psnr']:>10.3f} "
        f"{row['ssim']:>10.5f}"
    )


Loaded shapes:
X      : torch.Size([54750, 1, 13, 29])
hr_mask: torch.Size([54750, 1, 52, 116])
y      : torch.Size([54750, 1, 52, 116])
Training normalization split:
years      : 1951-2005
X_train   : torch.Size([20075, 1, 13, 29])
y_train   : torch.Size([20075, 1, 52, 116])
y_mean    : 5.909543037414551
y_std     : 10.123307228088379

Bilinear low_res -> high_res validation baseline
years        samples  normalized_mse   full_mse   land_mse  full_rmse  land_rmse       PSNR       SSIM
2006-2099      34310       0.3381692     34.656     41.371      5.887      6.432     23.566    0.61657
2006-2040      12775       0.3117271     31.946     39.704      5.652      6.301     23.527    0.60087
2041-2070      10950       0.3355852     34.391     41.084      5.864      6.410     22.944    0.60852
2071-2099      10585       0.3727551     38.200     43.679      6.181      6.609     22.170    0.61061
